# Test avec PCA 

## Objectif
Améliorer la précision du modèle en utilisant une PCA



## Import divers

In [2]:
## Import via notre python source

import sys
from pathlib import Path

# Le noyau Jupyter peut démarrer à la racine du dépôt ou dans notebooks/.
racine = Path.cwd()
while not (racine / "src" / "donnees.py").is_file():
    if racine == racine.parent:
        raise FileNotFoundError("Ouvrez le notebook depuis le dépôt ML.")
    racine = racine.parent
if str(racine) not in sys.path:
    sys.path.insert(0, str(racine))

from src.donnees import creer_dataloaders

## Import du dataset

In [3]:
train_loader, val_loader, test_loader = creer_dataloaders(batch_size=64)

print(f"Train : {len(train_loader.dataset)} images")
print(f"Validation : {len(val_loader.dataset)} images")
print(f"Test : {len(test_loader.dataset)} images")

Train : 24402 images
Validation : 4307 images
Test : 7178 images


## Vérification des lots

In [4]:
print("Train :", len(train_loader))
print("Validation :", len(val_loader))
print("Test :", len(test_loader))

Train : 382
Validation : 68
Test : 113


## Applatissement des images et conservation des étiquettes :

In [5]:
import torch

def extraire_images_et_labels(loader):
    vecteurs = []
    etiquettes = []

    for images, labels in loader:
        vecteurs.append(images.flatten(start_dim=1))
        etiquettes.append(labels)

    X = torch.cat(vecteurs, dim=0).numpy()
    y = torch.cat(etiquettes, dim=0)
    return X, y

X_train, y_train = extraire_images_et_labels(train_loader)
X_val, y_val = extraire_images_et_labels(val_loader)

print(X_train.shape)  # (24402, 2304)
print(X_val.shape)    # (4307, 2304)

(24402, 2304)
(4307, 2304)


## Ajuster la PCA sur l'entraînement

In [6]:
from sklearn.decomposition import PCA

pca = PCA(n_components=100, svd_solver="randomized", random_state=42)
pca.fit(X_train)

print(f"Variance conservée : {pca.explained_variance_ratio_.sum():.1%}")

Variance conservée : 89.8%


## Transformer l'entrainement et la validation avec PCA

In [7]:
X_train_pca = pca.transform(X_train)
X_val_pca = pca.transform(X_val)

print(X_train_pca.shape)  # (24402, 100)
print(X_val_pca.shape)    # (4307, 100)

(24402, 100)
(4307, 100)


## Recréer les lots et adapter le modèle

In [8]:
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

train_pca = TensorDataset(torch.from_numpy(X_train_pca).float(), y_train)
val_pca = TensorDataset(torch.from_numpy(X_val_pca).float(), y_val)

train_pca_loader = DataLoader(train_pca, batch_size=64, shuffle=True)
val_pca_loader = DataLoader(val_pca, batch_size=64, shuffle=False)

model = nn.Linear(pca.n_components_, 7)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [10]:
epochs = 20

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_pca_loader:
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_pca_loader):.4f}")

Epoch 1/20 - Loss: 1.8219
Epoch 2/20 - Loss: 1.6816
Epoch 3/20 - Loss: 1.6491
Epoch 4/20 - Loss: 1.6359
Epoch 5/20 - Loss: 1.6291
Epoch 6/20 - Loss: 1.6253
Epoch 7/20 - Loss: 1.6239
Epoch 8/20 - Loss: 1.6224
Epoch 9/20 - Loss: 1.6209
Epoch 10/20 - Loss: 1.6202
Epoch 11/20 - Loss: 1.6198
Epoch 12/20 - Loss: 1.6197
Epoch 13/20 - Loss: 1.6194
Epoch 14/20 - Loss: 1.6198
Epoch 15/20 - Loss: 1.6182
Epoch 16/20 - Loss: 1.6190
Epoch 17/20 - Loss: 1.6193
Epoch 18/20 - Loss: 1.6185
Epoch 19/20 - Loss: 1.6191
Epoch 20/20 - Loss: 1.6187


In [12]:
# Evaluation : 
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_pca_loader:
        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total

print(f"Validation accuracy : {accuracy:.2%}")

Validation accuracy : 37.15%
